In [3]:
import duckdb
con = duckdb.connect("amazon.duckdb")   # creates a local database file
con.execute("""
    CREATE TABLE reviews AS
    SELECT * FROM read_json_auto('data/raw/review_categories/Health_and_Personal_Care.jsonl')
""")
print(con.execute("SELECT COUNT(*) FROM reviews").fetchone())

(494121,)


In [5]:
con.execute("""
    CREATE TABLE products AS
    SELECT * FROM read_json_auto('data/raw/meta_categories/Health_and_Personal_Care.jsonl')
""")

con.sql("DESCRIBE reviews").show()
con.sql("DESCRIBE products").show()
con.sql("SUMMARIZE products").show()   # nulls, distinct counts, min/max per column

┌───────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │                                                  column_type                                                  │  null   │   key   │ default │  extra  │
│      varchar      │                                                    varchar                                                    │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ rating            │ DOUBLE                                                                                                        │ YES     │ NULL    │ NULL    │ NULL    │
│ title             │ VARCHAR                                                                                                     

In [8]:
# How many products have no price?
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE price IS NULL) AS no_price,
        COUNT(*) AS total,
        ROUND(100.0 * COUNT(*) FILTER (WHERE price IS NULL) / COUNT(*), 2) AS no_price_pct
    FROM products
""").show()

# Brand usually lives in the 'store' field; check its messiness
con.sql("SELECT store, COUNT(*) n FROM products GROUP BY 1 ORDER BY n DESC LIMIT 30").show()

# Duplicate reviews?
con.sql("""SELECT user_id, parent_asin, timestamp, COUNT(*) c
           FROM reviews GROUP BY ALL HAVING c > 1""").show()



┌──────────┬───────┬──────────────┐
│ no_price │ total │ no_price_pct │
│  int64   │ int64 │    double    │
├──────────┼───────┼──────────────┤
│    49757 │ 60293 │        82.53 │
└──────────┴───────┴──────────────┘

┌─────────────────────┬───────┐
│        store        │   n   │
│       varchar       │ int64 │
├─────────────────────┼───────┤
│ NULL                │  2346 │
│ Eyekepper           │   668 │
│ HAARBB              │   466 │
│ Generic             │   260 │
│ Pure Support        │   141 │
│ uxcell              │   104 │
│ Andaz Press         │   102 │
│ Unknown             │    92 │
│ Glade               │    78 │
│ Artist Unknown      │    77 │
│       ·             │     · │
│       ·             │     · │
│       ·             │     · │
│ Creative Converting │    57 │
│ Fun Express         │    55 │
│ TruVision Readers   │    55 │
│ SUNPRO              │    51 │
│ HomArt              │    51 │
│ Miraflex            │    50 │
│ EYEURL              │    50 │
│ Supportiback 

Almost 82.53% of Rows are missing price

In [10]:
con.sql("""
    SELECT p.price IS NOT NULL AS has_price,
           COUNT(*) AS reviews
    FROM reviews r
    JOIN products p USING (parent_asin)
    GROUP BY 1
""").show()

┌───────────┬─────────┐
│ has_price │ reviews │
│  boolean  │  int64  │
├───────────┼─────────┤
│ true      │  169019 │
│ false     │  325102 │
└───────────┴─────────┘



In [12]:
con.sql("""
SELECT p.price IS NOT NULL AS has_price,
       YEAR(to_timestamp(MEDIAN(r.timestamp)/1000)) AS median_review_year
FROM reviews r JOIN products p USING (parent_asin)
GROUP BY 1
""").show()

┌───────────┬────────────────────┐
│ has_price │ median_review_year │
│  boolean  │       int64        │
├───────────┼────────────────────┤
│ true      │               2019 │
│ false     │               2018 │
└───────────┴────────────────────┘



In [13]:
con.sql("""
SELECT categories[2] AS subcat, COUNT(*) n
FROM products GROUP BY 1 ORDER BY n DESC LIMIT 20
""").show()

┌────────┬───────┐
│ subcat │   n   │
│  json  │ int64 │
├────────┼───────┤
│ NULL   │ 60293 │
└────────┴───────┘



In [14]:
con.sql("SELECT categories, COUNT(*) n FROM products GROUP BY 1 ORDER BY n DESC LIMIT 5").show()
con.sql("SELECT main_category, COUNT(*) n FROM products GROUP BY 1 ORDER BY n DESC LIMIT 15").show()

┌────────────┬───────┐
│ categories │   n   │
│   json[]   │ int64 │
├────────────┼───────┤
│ []         │ 60293 │
└────────────┴───────┘

┌────────────────────────┬───────┐
│     main_category      │   n   │
│        varchar         │ int64 │
├────────────────────────┼───────┤
│ Health & Personal Care │ 60293 │
└────────────────────────┴───────┘



In [15]:
con.execute("""
CREATE OR REPLACE TABLE product_segment AS
SELECT parent_asin,
  CASE
    WHEN title ILIKE '%reading glasses%' OR title ILIKE '%readers%'     THEN 'Reading Glasses'
    WHEN title ILIKE '%brace%' OR title ILIKE '%support%'               THEN 'Braces & Supports'
    WHEN title ILIKE '%air freshener%' OR title ILIKE '%scented%'       THEN 'Air Fresheners'
    WHEN title ILIKE '%party%' OR title ILIKE '%favor%'                 THEN 'Party Supplies'
    WHEN title ILIKE '%thermometer%' OR title ILIKE '%blood pressure%'  THEN 'Health Monitors'
    ELSE 'Other'
  END AS segment
FROM products
""")

con.sql("""
SELECT s.segment, COUNT(DISTINCT s.parent_asin) AS products, COUNT(*) AS reviews
FROM product_segment s JOIN reviews r USING (parent_asin)
GROUP BY 1 ORDER BY reviews DESC
""").show()

┌───────────────────┬──────────┬─────────┐
│      segment      │ products │ reviews │
│      varchar      │  int64   │  int64  │
├───────────────────┼──────────┼─────────┤
│ Other             │    51975 │  442801 │
│ Braces & Supports │     3683 │   29262 │
│ Party Supplies    │     2368 │   11257 │
│ Air Fresheners    │      699 │    5154 │
│ Reading Glasses   │     1379 │    4195 │
│ Health Monitors   │      170 │    1452 │
└───────────────────┴──────────┴─────────┘



In [18]:
con.execute("""
CREATE OR REPLACE TABLE reviews_dedup AS
SELECT * FROM reviews
QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, parent_asin, timestamp
                           ORDER BY helpful_vote DESC) = 1
""")

con.sql("SELECT (SELECT COUNT(*) FROM reviews) AS before, (SELECT COUNT(*) FROM reviews_dedup) AS after").show()

┌────────┬────────┐
│ before │ after  │
│ int64  │ int64  │
├────────┼────────┤
│ 494121 │ 488990 │
└────────┴────────┘



In [19]:
con.execute("""
CREATE OR REPLACE TABLE brand_reviews AS
SELECT CASE WHEN p.store IS NULL
              OR LOWER(TRIM(p.store)) IN ('generic','unknown','artist unknown','n/a','na')
            THEN 'Unbranded' ELSE TRIM(p.store) END AS brand,
       r.rating,
       YEAR(to_timestamp(r.timestamp/1000)) AS yr
FROM reviews_dedup r JOIN products p USING (parent_asin)
""")

con.sql("""
SELECT brand, COUNT(*) AS reviews,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct,
       ROUND(AVG(rating), 2) AS avg_rating
FROM brand_reviews GROUP BY 1 ORDER BY reviews DESC LIMIT 20
""").show()

┌─────────────────┬─────────┬───────────┬────────────┐
│      brand      │ reviews │ share_pct │ avg_rating │
│     varchar     │  int64  │  double   │   double   │
├─────────────────┼─────────┼───────────┼────────────┤
│ Unbranded       │   13942 │      2.85 │       3.71 │
│ ASUTRA          │    3147 │      0.64 │       4.68 │
│ GermGuardian    │    2882 │      0.59 │       3.84 │
│ US Organic      │    2847 │      0.58 │        4.3 │
│ Purple          │    2750 │      0.56 │       4.05 │
│ Avon            │    2320 │      0.47 │       4.28 │
│ Carlson         │    2300 │      0.47 │       4.66 │
│ Essential Depot │    2187 │      0.45 │       4.73 │
│ Nerdwax         │    1990 │      0.41 │       3.43 │
│ Fitbit          │    1987 │      0.41 │       3.19 │
│ Mr. Clean       │    1925 │      0.39 │       4.35 │
│ Homedics        │    1747 │      0.36 │       3.49 │
│ Sano Naturals   │    1701 │      0.35 │       3.69 │
│ Clear Care      │    1649 │      0.34 │       4.14 │
│ JUNP    

In [20]:
con.sql("SELECT yr, COUNT(*) FROM brand_reviews GROUP BY 1 ORDER BY 1").show()

┌───────┬──────────────┐
│  yr   │ count_star() │
│ int64 │    int64     │
├───────┼──────────────┤
│  2001 │            6 │
│  2002 │            7 │
│  2003 │           14 │
│  2004 │           28 │
│  2005 │           60 │
│  2006 │           96 │
│  2007 │          333 │
│  2008 │          519 │
│  2009 │          756 │
│  2010 │         1478 │
│  2011 │         2122 │
│  2012 │         4016 │
│  2013 │         9906 │
│  2014 │        17384 │
│  2015 │        33964 │
│  2016 │        52813 │
│  2017 │        57102 │
│  2018 │        51881 │
│  2019 │        57644 │
│  2020 │        75901 │
│  2021 │        68446 │
│  2022 │        43262 │
│  2023 │        11252 │
└───────┴──────────────┘
  23 rows    2 columns



In [21]:
con.sql("""
SELECT brand,
       COUNT(*) FILTER (WHERE yr = 2021) AS rev_prev,
       COUNT(*) FILTER (WHERE yr = 2022) AS rev_last,
       ROUND(100.0 * (rev_last - rev_prev) / NULLIF(rev_prev,0), 1) AS growth_pct,
       ROUND(AVG(rating), 2) AS avg_rating
FROM brand_reviews
WHERE brand <> 'Unbranded'
GROUP BY 1
HAVING rev_prev >= 30
ORDER BY growth_pct DESC
""").show(max_rows=40)

┌─────────────────────────────┬──────────┬──────────┬────────────┬────────────┐
│            brand            │ rev_prev │ rev_last │ growth_pct │ avg_rating │
│           varchar           │  int64   │  int64   │   double   │   double   │
├─────────────────────────────┼──────────┼──────────┼────────────┼────────────┤
│ PEFOOK                      │       33 │      134 │      306.1 │       4.03 │
│ Mini Thin                   │       35 │      124 │      254.3 │       3.44 │
│ Hion                        │       61 │      213 │      249.2 │       3.04 │
│ Bridgewater Candle          │       43 │      118 │      174.4 │       4.19 │
│ DownBeats                   │       33 │       87 │      163.6 │       4.19 │
│ Lingito                     │       38 │      100 │      163.2 │       3.92 │
│ Gavana                      │       36 │       84 │      133.3 │       2.36 │
│ HeelTastic                  │       34 │       67 │       97.1 │       4.53 │
│ Cascade                     │       32

In [22]:
con.sql("""
WITH yearly AS (
  SELECT brand, yr, COUNT(*) AS n,
         COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY yr) AS share
  FROM brand_reviews
  WHERE yr IN (2021, 2022)
  GROUP BY brand, yr
)
SELECT b.brand,
       MAX(CASE WHEN yr=2021 THEN n END) AS rev_2021,
       MAX(CASE WHEN yr=2022 THEN n END) AS rev_2022,
       ROUND(100 * (MAX(CASE WHEN yr=2022 THEN share END) /
                    NULLIF(MAX(CASE WHEN yr=2021 THEN share END),0) - 1), 1) AS share_growth_pct,
       ROUND(r.avg_rating, 2) AS avg_rating
FROM yearly b
JOIN (SELECT brand, AVG(rating) AS avg_rating FROM brand_reviews GROUP BY 1) r USING (brand)
WHERE b.brand <> 'Unbranded'
GROUP BY b.brand, r.avg_rating
HAVING MIN(n) >= 50
ORDER BY share_growth_pct DESC
""").show(max_rows=40)

┌───────────────────────┬──────────┬──────────┬──────────────────┬────────────┐
│         brand         │ rev_2021 │ rev_2022 │ share_growth_pct │ avg_rating │
│        varchar        │  int64   │  int64   │      double      │   double   │
├───────────────────────┼──────────┼──────────┼──────────────────┼────────────┤
│ Hion                  │       61 │      213 │            452.4 │       3.04 │
│ Miyuki                │       53 │      101 │            201.5 │       3.21 │
│ Natural Armor         │       74 │      128 │            173.7 │       2.31 │
│ The Honey Pot Company │       97 │      157 │            156.1 │       3.69 │
│ 10 Seconds            │       64 │       98 │            142.3 │       3.92 │
│ Gold Bond             │      140 │      209 │            136.2 │       3.32 │
│ Systane               │       64 │       86 │            112.6 │       4.29 │
│ BIZ                   │       80 │      105 │            107.7 │       4.63 │
│ Faraone4w             │      173 │    

In [23]:
con.sql("""
SELECT p.store, p.title, COUNT(*) AS reviews, ROUND(AVG(r.rating),2) AS avg_rating
FROM reviews_dedup r JOIN products p USING (parent_asin)
WHERE p.store IN ('Gavana','Natural Armor','Hion','Miyuki','Nature''s MACE')
GROUP BY 1,2 ORDER BY reviews DESC LIMIT 20
""").show(max_width=200)

┌───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬────────────┐
│     store     │                                                                             title                                                                             │ reviews │ avg_rating │
│    varchar    │                                                                            varchar                                                                            │  int64  │   double   │
├───────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼────────────┤
│ Nature's MACE │ Nature's MACE Cat Repellent 1 Gallon Spray/Treats 3,000 Sq. Ft. / Keep Cat Out of Your Lawn and Garden/Train Your Cat to Stay Out of Bushes/Safe to use Arou… │    1527 │       2.

In [24]:
con.sql("""
SELECT CASE WHEN p.title ILIKE '%repellent%' OR p.title ILIKE '%deterrent%' THEN 'Pest Repellents'
            WHEN p.title ILIKE '%motion sickness%' OR p.title ILIKE '%nausea%'  THEN 'Motion Sickness'
       END AS market,
       COUNT(DISTINCT p.parent_asin) AS products,
       COUNT(DISTINCT p.store)       AS brands,
       COUNT(*)                      AS reviews,
       ROUND(AVG(r.rating), 2)       AS avg_rating,
       ROUND(100.0 * AVG(CASE WHEN r.rating <= 2 THEN 1 ELSE 0 END), 1) AS pct_negative
FROM reviews_dedup r JOIN products p USING (parent_asin)
WHERE market IS NOT NULL
GROUP BY 1
""").show()

┌─────────────────┬──────────┬────────┬─────────┬────────────┬──────────────┐
│     market      │ products │ brands │ reviews │ avg_rating │ pct_negative │
│     varchar     │  int64   │ int64  │  int64  │   double   │    double    │
├─────────────────┼──────────┼────────┼─────────┼────────────┼──────────────┤
│ Pest Repellents │       82 │     62 │    5199 │       3.56 │         31.2 │
│ Motion Sickness │       55 │     43 │     488 │       3.61 │         30.9 │
└─────────────────┴──────────┴────────┴─────────┴────────────┴──────────────┘



In [26]:
con.sql("""
SELECT p.store, r.rating, r.text
FROM reviews_dedup r JOIN products p USING (parent_asin)
WHERE r.rating <= 2
  AND (p.title ILIKE '%repellent%' OR p.title ILIKE '%deterrent%')
ORDER BY random()
LIMIT 30
""").show(max_width=250)

┌───────────────────────┬────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│         store         │ rating │                                                                                                         text                                                                                                          │
│        varchar        │ double │                                                                                                        varchar                                                                                                        │
├───────────────────────┼────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [27]:
con.sql("""
WITH neg AS (
  SELECT LOWER(r.text) AS t
  FROM reviews_dedup r JOIN products p USING (parent_asin)
  WHERE r.rating <= 2
    AND (p.title ILIKE '%repellent%' OR p.title ILIKE '%deterrent%')
    AND p.title NOT ILIKE '%bandage%' AND p.title NOT ILIKE '%color%'
)
SELECT COUNT(*) AS neg_reviews,
  ROUND(100*AVG(CASE WHEN t SIMILAR TO '.*(not work|didn.t work|doesn.t work|does not work|no effect|useless|waste).*' THEN 1 ELSE 0 END),1) AS pct_ineffective,
  ROUND(100*AVG(CASE WHEN t SIMILAR TO '.*(minutes|hours|days|wore off|wears off|stopped working|reapply|washed away|rain).*' THEN 1 ELSE 0 END),1) AS pct_short_lasting,
  ROUND(100*AVG(CASE WHEN t SIMILAR TO '.*(smell|stink|odor|stench).*' THEN 1 ELSE 0 END),1) AS pct_smell,
  ROUND(100*AVG(CASE WHEN t SIMILAR TO '.*(leak|broke|spray|nozzle|hose|bottle|clog).*' THEN 1 ELSE 0 END),1) AS pct_packaging,
  ROUND(100*AVG(CASE WHEN t SIMILAR TO '.*(price|expensive|overpriced|refund|return).*' THEN 1 ELSE 0 END),1) AS pct_value
FROM neg
""").show()

┌─────────────┬─────────────────┬───────────────────┬───────────┬───────────────┬───────────┐
│ neg_reviews │ pct_ineffective │ pct_short_lasting │ pct_smell │ pct_packaging │ pct_value │
│    int64    │     double      │      double       │  double   │    double     │  double   │
├─────────────┼─────────────────┼───────────────────┼───────────┼───────────────┼───────────┤
│        1602 │            40.4 │               9.9 │      16.2 │          30.3 │       9.1 │
└─────────────┴─────────────────┴───────────────────┴───────────┴───────────────┴───────────┘



In [28]:
con.sql("""
WITH neg AS (
  SELECT LOWER(r.text) AS t
  FROM reviews_dedup r JOIN products p USING (parent_asin)
  WHERE r.rating <= 2
    AND (p.title ILIKE '%repellent%' OR p.title ILIKE '%deterrent%')
    AND p.title NOT ILIKE '%bandage%' AND p.title NOT ILIKE '%color%'
),
flags AS (
  SELECT t,
    t SIMILAR TO '.*(not work|didn.t work|doesn.t work|does not work|no effect|useless|waste|still (come|coming|there|around|here|see)|didn.t (deter|repel|keep|stop)|did not (deter|repel|keep|stop)|no difference).*' AS ineffective,
    t SIMILAR TO '.*(wore off|wears off|stopped working|only lasts|few (minutes|hours|days)|reapply|washed away|after it rain).*' AS short_lasting,
    t SIMILAR TO '.*(smell|stink|odor|stench).*' AS smell,
    t SIMILAR TO '.*(leak|broke|broken|nozzle|clog|hose|trigger).*' AS packaging,
    t SIMILAR TO '.*(price|expensive|overpriced|refund|return).*' AS value
  FROM neg
)
SELECT COUNT(*) AS neg_reviews,
  ROUND(100*AVG(ineffective::INT),1)   AS pct_ineffective,
  ROUND(100*AVG(short_lasting::INT),1) AS pct_short_lasting,
  ROUND(100*AVG(smell::INT),1)         AS pct_smell,
  ROUND(100*AVG(packaging::INT),1)     AS pct_packaging,
  ROUND(100*AVG(value::INT),1)         AS pct_value,
  ROUND(100*AVG((NOT (ineffective OR short_lasting OR smell OR packaging OR value))::INT),1) AS pct_unclassified
FROM flags
""").show()

┌─────────────┬─────────────────┬───────────────────┬───────────┬───────────────┬───────────┬──────────────────┐
│ neg_reviews │ pct_ineffective │ pct_short_lasting │ pct_smell │ pct_packaging │ pct_value │ pct_unclassified │
│    int64    │     double      │      double       │  double   │    double     │  double   │      double      │
├─────────────┼─────────────────┼───────────────────┼───────────┼───────────────┼───────────┼──────────────────┤
│        1602 │            46.0 │               2.2 │      16.2 │           4.9 │       9.1 │             37.2 │
└─────────────┴─────────────────┴───────────────────┴───────────┴───────────────┴───────────┴──────────────────┘



In [29]:
con.sql("""
WITH neg AS (
  SELECT LOWER(r.text) AS t
  FROM reviews_dedup r JOIN products p USING (parent_asin)
  WHERE r.rating <= 2
    AND (p.title ILIKE '%repellent%' OR p.title ILIKE '%deterrent%')
    AND p.title NOT ILIKE '%bandage%' AND p.title NOT ILIKE '%color%'
)
SELECT t FROM neg
WHERE NOT regexp_matches(t, 'not work|didn.t work|doesn.t work|does not work|no effect|useless|waste|still (come|coming|there|around|here|see)|didn.t (deter|repel|keep|stop)|did not (deter|repel|keep|stop)|no difference|wore off|wears off|stopped working|only lasts|few (minutes|hours|days)|reapply|washed away|after it rain|smell|stink|odor|stench|leak|broke|broken|nozzle|clog|hose|trigger|price|expensive|overpriced|refund|return')
ORDER BY random() LIMIT 40
""").show(max_width=250, max_rows=40)

┌────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                                                                                           t                                                                                                                            │
│                                                                                                                        varchar                                                                                                                         │
├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [30]:
con.sql("""
WITH neg AS (
  SELECT regexp_replace(
           regexp_replace(LOWER(r.text), '<br\\s*/?>', ' ', 'g'),
           '[''’]', '', 'g') AS t
  FROM reviews_dedup r JOIN products p USING (parent_asin)
  WHERE r.rating <= 2
    AND (p.title ILIKE '%repellent%' OR p.title ILIKE '%deterrent%')
    AND p.title NOT ILIKE '%bandage%' AND p.title NOT ILIKE '%color%'
),
flags AS (
  SELECT t,
    regexp_matches(t, 'not work|didnt work|doesnt work|dont work|does not work|no effect|useless|waste|did nothing|does nothing|didnt do anything|doesnt do anything|not do anything|nothing!|ignore|didnt mind|didnt budge|not (deter|repel|protect|stop|keep)|didnt (deter|repel|keep|stop|help)|hasnt (stopped|deterred|worked)|has not (stopped|deterred|worked)|still (come|coming|there|around|here|see)|no difference|attract|more (cats|bugs|lizards|mice)|getting bit|eaten alive|scam|no funciona|sin resultados') AS ineffective,
    regexp_matches(t, 'wore off|wears off|stopped working|only lasts|few (minutes|hours|days)|reapply|washed away|after it rain') AS short_lasting,
    regexp_matches(t, 'smell|stink|odor|stench|fragrance|aroma|overpowering') AS smell,
    regexp_matches(t, 'fake|diluted|counterfeit|curdled|watery|expired|expiration|half full|damaged|old product|seemed old') AS quality,
    regexp_matches(t, 'stain|died|killed my|sting|burn|rash|irritat|stomach') AS side_effects,
    regexp_matches(t, 'leak|broke|broken|nozzle|clog|hose|trigger|sprayer') AS packaging,
    regexp_matches(t, 'price|expensive|overpriced|ridiculous|refund|return|cuesta') AS value
  FROM neg
)
SELECT COUNT(*) AS neg_reviews,
  ROUND(100*AVG(ineffective::INT),1)   AS pct_ineffective,
  ROUND(100*AVG(short_lasting::INT),1) AS pct_short_lasting,
  ROUND(100*AVG(smell::INT),1)         AS pct_smell,
  ROUND(100*AVG(quality::INT),1)       AS pct_quality,
  ROUND(100*AVG(side_effects::INT),1)  AS pct_side_effects,
  ROUND(100*AVG(packaging::INT),1)     AS pct_packaging,
  ROUND(100*AVG(value::INT),1)         AS pct_value,
  ROUND(100*AVG((NOT (ineffective OR short_lasting OR smell OR quality OR side_effects OR packaging OR value))::INT),1) AS pct_unclassified
FROM flags
""").show()

┌─────────────┬─────────────────┬───────────────────┬───────────┬─────────────┬──────────────────┬───────────────┬───────────┬──────────────────┐
│ neg_reviews │ pct_ineffective │ pct_short_lasting │ pct_smell │ pct_quality │ pct_side_effects │ pct_packaging │ pct_value │ pct_unclassified │
│    int64    │     double      │      double       │  double   │   double    │      double      │    double     │  double   │      double      │
├─────────────┼─────────────────┼───────────────────┼───────────┼─────────────┼──────────────────┼───────────────┼───────────┼──────────────────┤
│        1602 │            55.3 │               2.2 │      16.8 │         2.1 │              5.4 │           6.5 │       9.2 │             25.7 │
└─────────────┴─────────────────┴───────────────────┴───────────┴─────────────┴──────────────────┴───────────────┴───────────┴──────────────────┘



In [31]:
import os
os.makedirs("exports", exist_ok=True)

exports = {
  # Page 1: category overview
  "yearly_reviews": "SELECT yr, COUNT(*) AS reviews FROM brand_reviews GROUP BY 1 ORDER BY 1",
  "brand_share": """SELECT brand, COUNT(*) AS reviews,
                      ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) AS share_pct,
                      ROUND(AVG(rating),2) AS avg_rating
                    FROM brand_reviews GROUP BY 1 ORDER BY reviews DESC LIMIT 50""",
  # Page 2: pest repellent market
  "pest_reviews": """SELECT p.store AS brand, p.title, r.rating,
                       to_timestamp(r.timestamp/1000) AS review_date
                     FROM reviews_dedup r JOIN products p USING (parent_asin)
                     WHERE (p.title ILIKE '%repellent%' OR p.title ILIKE '%deterrent%')
                       AND p.title NOT ILIKE '%bandage%' AND p.title NOT ILIKE '%color%'""",
}

for name, q in exports.items():
    con.sql(q).write_csv(f"exports/{name}.csv")
    print(name, "saved")

yearly_reviews saved
brand_share saved
pest_reviews saved


In [32]:
import os
os.makedirs("exports", exist_ok=True)

# ---------- 1. Brand opportunity ----------
con.sql("""
WITH yearly AS (
  SELECT brand, yr, COUNT(*) AS n,
         COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY yr) AS share
  FROM brand_reviews
  WHERE yr IN (2021, 2022)
  GROUP BY brand, yr
)
SELECT b.brand,
       MAX(CASE WHEN yr=2021 THEN n END) AS rev_2021,
       MAX(CASE WHEN yr=2022 THEN n END) AS rev_2022,
       ROUND(100 * (MAX(CASE WHEN yr=2022 THEN share END) /
                    NULLIF(MAX(CASE WHEN yr=2021 THEN share END),0) - 1), 1) AS share_growth_pct,
       ROUND(r.avg_rating, 2) AS avg_rating
FROM yearly b
JOIN (SELECT brand, AVG(rating) AS avg_rating FROM brand_reviews GROUP BY 1) r USING (brand)
WHERE b.brand <> 'Unbranded'
GROUP BY b.brand, r.avg_rating
HAVING MIN(n) >= 50 AND COUNT(*) = 2
ORDER BY share_growth_pct DESC
""").write_csv("exports/brand_opportunity.csv")

# ---------- 2. Complaint themes ----------
themes_wide = con.sql("""
WITH neg AS (
  SELECT regexp_replace(
           regexp_replace(LOWER(r.text), '<br\\s*/?>', ' ', 'g'),
           '[''’]', '', 'g') AS t
  FROM reviews_dedup r JOIN products p USING (parent_asin)
  WHERE r.rating <= 2
    AND (p.title ILIKE '%repellent%' OR p.title ILIKE '%deterrent%')
    AND p.title NOT ILIKE '%bandage%' AND p.title NOT ILIKE '%color%'
),
flags AS (
  SELECT
    regexp_matches(t, 'not work|didnt work|doesnt work|dont work|does not work|no effect|useless|waste|did nothing|does nothing|didnt do anything|doesnt do anything|not do anything|nothing!|ignore|didnt mind|didnt budge|not (deter|repel|protect|stop|keep)|didnt (deter|repel|keep|stop|help)|hasnt (stopped|deterred|worked)|has not (stopped|deterred|worked)|still (come|coming|there|around|here|see)|no difference|attract|more (cats|bugs|lizards|mice)|getting bit|eaten alive|scam|no funciona|sin resultados') AS ineffective,
    regexp_matches(t, 'wore off|wears off|stopped working|only lasts|few (minutes|hours|days)|reapply|washed away|after it rain') AS short_lasting,
    regexp_matches(t, 'smell|stink|odor|stench|fragrance|aroma|overpowering') AS smell,
    regexp_matches(t, 'fake|diluted|counterfeit|curdled|watery|expired|expiration|half full|damaged|old product|seemed old') AS quality,
    regexp_matches(t, 'stain|died|killed my|sting|burn|rash|irritat|stomach') AS side_effects,
    regexp_matches(t, 'leak|broke|broken|nozzle|clog|hose|trigger|sprayer') AS packaging,
    regexp_matches(t, 'price|expensive|overpriced|ridiculous|refund|return|cuesta') AS value
  FROM neg
)
SELECT
  ROUND(100*AVG(ineffective::INT),1)   AS "Ineffective",
  ROUND(100*AVG(smell::INT),1)         AS "Mentions Smell",
  ROUND(100*AVG(value::INT),1)         AS "Price / Value",
  ROUND(100*AVG(packaging::INT),1)     AS "Packaging",
  ROUND(100*AVG(side_effects::INT),1)  AS "Side Effects / Damage",
  ROUND(100*AVG(short_lasting::INT),1) AS "Short-Lasting",
  ROUND(100*AVG(quality::INT),1)       AS "Quality / Authenticity",
  ROUND(100*AVG((NOT (ineffective OR short_lasting OR smell OR quality
                      OR side_effects OR packaging OR value))::INT),1) AS "Unclassified"
FROM flags
""").df()

themes_long = (themes_wide
               .melt(var_name="theme", value_name="pct_of_negative_reviews")
               .sort_values("pct_of_negative_reviews", ascending=False))
themes_long.to_csv("exports/complaint_themes.csv", index=False)

print(themes_long)
print(os.listdir("exports"))

                    theme  pct_of_negative_reviews
0             Ineffective                     55.3
7            Unclassified                     25.7
1          Mentions Smell                     16.8
2           Price / Value                      9.2
3               Packaging                      6.5
4   Side Effects / Damage                      5.4
5           Short-Lasting                      2.2
6  Quality / Authenticity                      2.1
['pest_reviews.csv', 'brand_share.csv', 'yearly_reviews.csv', 'complaint_themes.csv', 'brand_opportunity.csv']
